# Module 5.1: Deploy the Grounded Booking Agent to AgentCore Runtime

The agent from Module 3.2 has run in a notebook kernel on your laptop. Your
laptop holds the Neo4j password, your credentials make the Bedrock call, and
the only way to ask the agent a question is to open Jupyter.

Here you move that same agent, unchanged in how it reasons, into a container
that :link[Amazon Bedrock AgentCore Runtime]{href="https://aws.amazon.com/bedrock/agentcore/" external=true}
starts and holds. What changes is operational, not behavioural:

| Module 3.2 | Module 5.1 |
|---|---|
| Runs in your kernel | Runs in a container AgentCore starts |
| Your laptop holds the Neo4j password | The Runtime holds it, injected at launch |
| Reachable only from Jupyter | Reachable by `InvokeAgentRuntime` from anywhere |
| Session is your kernel's memory | Each invocation is isolated by session ID |

Both tools stay in-process against Neo4j. The reservation rule lives in the
graph and is enforced in the same transaction as the write, so moving the agent
into a container does not move the rule.

:::alert{type="warning" header="AWS resources created"}
One IAM execution role, one ECR repository, one CodeBuild project, and one
AgentCore Runtime. The build takes three to five minutes. The cleanup notebook
removes all four.
:::

In [ ]:
import sys, os
# Add notebooks/ root to path so the shared `workshop` package is importable
_notebooks_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if _notebooks_root not in sys.path:
    sys.path.insert(0, _notebooks_root)

## 1. Confirm what this deploy needs

The Runtime reads its Neo4j connection from environment variables injected at
launch, so the same four values every other module uses have to be present here
before the container is built.

In [ ]:
import json
import shutil
import subprocess
import uuid
from datetime import date, timedelta
from pathlib import Path

import boto3
from dotenv import load_dotenv, find_dotenv

from workshop.aws_region import configure_aws_region
from workshop.bedrock_providers import default_model_id

load_dotenv(find_dotenv())

REPO_ROOT = Path(_notebooks_root).parent
REGION = configure_aws_region()
MODEL_ID = default_model_id()

# Every name this deploy creates, derived from one prefix. The starter toolkit
# derives the ECR repository and CodeBuild project names from the Runtime name,
# so cleanup can only find what this notebook created if it derives them the
# same way. AgentCore Runtime names accept letters, digits and underscores
# only, which is why this one carries no hyphen.
RUNTIME_NAME = "GraphRagBookingAgent"
ROLE_NAME = "workshop-graphrag-runtime-role"
ECR_REPO = "workshop-graphrag-booking-agent"
CB_PROJECT = f"bedrock-agentcore-{RUNTIME_NAME.lower()}-builder"

# The teardown gate. Cleanup deletes a resource only if it carries this exact
# key and value, and each of the three services below demands its own shape.
WORKSHOP_TAG_KEY = "WorkshopResource"
WORKSHOP_TAG_VALUE = "graphrag-with-neo4j"
WORKSHOP_TAGS_MAP = {WORKSHOP_TAG_KEY: WORKSHOP_TAG_VALUE}                          # agentcore
WORKSHOP_TAGS_KV = [{"Key": WORKSHOP_TAG_KEY, "Value": WORKSHOP_TAG_VALUE}]         # ecr, iam
WORKSHOP_TAGS_KV_LOWER = [{"key": WORKSHOP_TAG_KEY, "value": WORKSHOP_TAG_VALUE}]   # codebuild

# The deployed Runtime reads its Neo4j connection from these, so they have to be
# forwarded as container environment variables at launch.
NEO4J_ENV = ("NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD", "NEO4J_DATABASE")
NEO4J_VALUES = {name: os.getenv(name, "").strip() for name in NEO4J_ENV}

AWS_READY = boto3.Session().get_credentials() is not None
missing = [name for name, value in NEO4J_VALUES.items() if not value]
DEPLOY_READY = AWS_READY and not missing

print(f"region:          {REGION}")
print(f"model:           {MODEL_ID}")
print(f"runtime name:    {RUNTIME_NAME}")
print(f"execution role:  {ROLE_NAME}")
print(f"AWS credentials: {'found' if AWS_READY else 'NOT FOUND'}")
print(f"Neo4j values:    {'all four present' if not missing else 'missing ' + ', '.join(missing)}")

if not DEPLOY_READY:
    print("\nNot ready to deploy. Every live cell below will skip.")
    if missing:
        print(f"  Add to {REPO_ROOT / '.env'}: {', '.join(missing)}")
    if not AWS_READY:
        print("  Configure AWS credentials before re-running this cell.")
else:
    print("\nReady to deploy.")

## 2. Stage the shared code into the build context

Docker copies only the build context, and both halves of this agent live
outside it. `workshop/` is the package every module shares, and
`reservation_command.py` is the graph-enforced write path Module 3.2 calls.
Staging them here keeps one copy of each in version control instead of a second
copy that drifts.

The staged copies are gitignored and overwritten on every run, so an edit to
either source file reaches the next build.

In [ ]:
DEPLOY_DIR = Path.cwd() / "runtime_app"
if not DEPLOY_DIR.is_dir():
    raise FileNotFoundError(
        "runtime_app/ not found. Run this notebook from 05-agentcore-deploy/."
    )

PACKAGE_SRC = Path(_notebooks_root) / "workshop"
COMMAND_SRC = Path(_notebooks_root) / "03-retrieval-patterns" / "reservation_command.py"

# Staged unconditionally, even when the deploy will skip. It writes only inside
# runtime_app/, it is cheap, and it means a participant without AWS
# credentials can still read exactly what would have gone into the image.
staged_package = DEPLOY_DIR / "workshop"
staged_command = DEPLOY_DIR / "reservation_command.py"

# Removed and rewritten rather than merged. A file deleted from workshop/ would
# otherwise survive in the staged copy and keep satisfying an import that the
# real package no longer serves, which builds cleanly and fails in the
# container.
if staged_package.exists():
    shutil.rmtree(staged_package)
shutil.copytree(
    PACKAGE_SRC,
    staged_package,
    ignore=shutil.ignore_patterns("__pycache__", "*.pyc"),
)
shutil.copy2(COMMAND_SRC, staged_command)

# Recorded into the build context itself, not just printed here. A printed
# commit lives in notebook output, which does not survive into the image;
# a file inside the build context does, so a rebuilt image can be checked
# against its own claim of what it was built from.
git_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, capture_output=True, text=True, check=True
).stdout.strip()
git_dirty = bool(
    subprocess.run(
        ["git", "status", "--porcelain"], cwd=REPO_ROOT, capture_output=True, text=True, check=True
    ).stdout.strip()
)
(DEPLOY_DIR / "BUILD_INFO.txt").write_text(f"commit={git_commit}\ndirty={git_dirty}\n")

print(f"\nBuild commit: {git_commit}{'  (DIRTY TREE)' if git_dirty else ''}")
if git_dirty:
    print(
        "WARNING: uncommitted changes are present. This image will not trace to a\n"
        "git ref. Commit before deploying if this build needs to be reproducible."
    )

staged_files = sorted(p for p in staged_package.rglob("*.py"))
print(f"Staged {staged_package.relative_to(Path.cwd())}/ ({len(staged_files)} modules)")
print(f"Staged {staged_command.relative_to(Path.cwd())}")
print("\nBuild context now contains:")
for path in sorted(DEPLOY_DIR.iterdir()):
    marker = "/" if path.is_dir() else ""
    print(f"  {path.name}{marker}")

## 3. Create the Runtime execution role

AgentCore assumes this role inside the container. It grants image pulls, the
Runtime log groups, X-Ray, the workload-identity tokens, and Bedrock model
invocation. It grants no Neo4j access, because Neo4j is not an AWS service. The
graph credentials arrive as environment variables at launch.

Bedrock invocation is scoped to `foundation-model/*` plus the account-scoped
ARN rather than one model ARN. The workshop model is a cross-region inference
profile, which fans out to several foundation-model ARNs, so pinning a single
ARN would deny the call.

In [ ]:
ROLE_ARN = ""

if not DEPLOY_READY:
    print("Skipping role creation: see Step 1.")
else:
    iam = boto3.client("iam")
    account_id = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]
    runtime_logs = f"arn:aws:logs:{REGION}:{account_id}:log-group:/aws/bedrock-agentcore/runtimes/"

    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Sid": "AssumeRolePolicy",
                "Effect": "Allow",
                "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
                "Action": "sts:AssumeRole",
                # Scoped to this account and this service. Without the
                # condition the role is assumable by AgentCore in any account
                # that learns its ARN, which is the confused-deputy shape the
                # service documentation warns about.
                "Condition": {
                    "StringEquals": {"aws:SourceAccount": account_id},
                    "ArnLike": {
                        "aws:SourceArn": f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:*"
                    },
                },
            }
        ],
    }

    inline_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Sid": "ECRImageAccess",
                "Effect": "Allow",
                "Action": ["ecr:BatchGetImage", "ecr:GetDownloadUrlForLayer"],
                "Resource": f"arn:aws:ecr:{REGION}:{account_id}:repository/*",
            },
            {
                "Sid": "ECRTokenAccess",
                "Effect": "Allow",
                "Action": ["ecr:GetAuthorizationToken"],
                "Resource": "*",
            },
            {
                "Sid": "RuntimeLogs",
                "Effect": "Allow",
                "Action": [
                    "logs:CreateLogGroup",
                    "logs:CreateLogStream",
                    "logs:PutLogEvents",
                    "logs:DescribeLogStreams",
                ],
                "Resource": f"{runtime_logs}*",
            },
            {
                "Sid": "DescribeLogGroups",
                "Effect": "Allow",
                "Action": ["logs:DescribeLogGroups"],
                "Resource": f"arn:aws:logs:{REGION}:{account_id}:log-group:*",
            },
            {
                "Sid": "XRay",
                "Effect": "Allow",
                "Action": [
                    "xray:PutTraceSegments",
                    "xray:PutTelemetryRecords",
                    "xray:GetSamplingRules",
                    "xray:GetSamplingTargets",
                ],
                "Resource": "*",
            },
            {
                "Sid": "CloudWatchMetrics",
                "Effect": "Allow",
                "Action": "cloudwatch:PutMetricData",
                "Resource": "*",
                "Condition": {
                    "StringEquals": {"cloudwatch:namespace": "bedrock-agentcore"}
                },
            },
            {
                "Sid": "GetAgentAccessToken",
                "Effect": "Allow",
                "Action": [
                    "bedrock-agentcore:GetWorkloadAccessToken",
                    "bedrock-agentcore:GetWorkloadAccessTokenForJWT",
                    "bedrock-agentcore:GetWorkloadAccessTokenForUserId",
                ],
                "Resource": [
                    f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:workload-identity-directory/default",
                    f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:workload-identity-directory/default/workload-identity/*",
                ],
            },
            {
                "Sid": "BedrockModelInvocation",
                "Effect": "Allow",
                "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"],
                "Resource": [
                    "arn:aws:bedrock:*::foundation-model/*",
                    f"arn:aws:bedrock:{REGION}:{account_id}:*",
                ],
            },
        ],
    }

    # create_role then fall back to update. A blind delete-and-recreate would
    # break any Runtime already using the role, and re-running this notebook
    # should be safe.
    try:
        iam.create_role(
            RoleName=ROLE_NAME,
            AssumeRolePolicyDocument=json.dumps(trust_policy),
            Description="AgentCore Runtime execution role for the GraphRAG workshop",
            Tags=WORKSHOP_TAGS_KV,
        )
        print(f"Created role: {ROLE_NAME}")
    except iam.exceptions.EntityAlreadyExistsException:
        iam.update_assume_role_policy(
            RoleName=ROLE_NAME, PolicyDocument=json.dumps(trust_policy)
        )
        print(f"Role already exists, trust policy refreshed: {ROLE_NAME}")

    iam.put_role_policy(
        RoleName=ROLE_NAME,
        PolicyName="graphrag-runtime-policy",
        PolicyDocument=json.dumps(inline_policy),
    )
    ROLE_ARN = iam.get_role(RoleName=ROLE_NAME)["Role"]["Arn"]
    print(f"Execution role: {ROLE_ARN}")

## 4. Configure and launch

The starter toolkit builds the image in CodeBuild, pushes it to ECR, and
creates the Runtime. It uses the current directory as the build root, honouring
the `Dockerfile` it finds there, so this cell changes into `runtime_app/`
first. Run it from the module folder instead and the toolkit writes its own
Dockerfile and ships the whole folder.

Expect three to five minutes.

In [ ]:
RUNTIME_ARN = ""
RUNTIME_ID = None

if not DEPLOY_READY:
    print("Skipping launch: see Step 1.")
else:
    from bedrock_agentcore_starter_toolkit import Runtime

    if Path.cwd() != DEPLOY_DIR:
        os.chdir(DEPLOY_DIR)
    print(f"Build context: {Path.cwd()}")

    # Remove only the config file this notebook's toolkit run writes, in this
    # directory. It records the runtime ID of a previous deploy, and a stale one
    # sends the launch at a Runtime that may no longer exist.
    local_config = Path.cwd() / ".bedrock_agentcore.yaml"
    if local_config.exists():
        local_config.unlink()
        print(f"Removed stale toolkit config: {local_config.name}")

    # The participant IAM policy only grants ECR actions on repositories named
    # workshop-*, so the repository is created here under a name that policy
    # allows. Left to auto_create_ecr, the toolkit derives the name from the
    # runtime and produces one the policy denies, which fails the push for a
    # participant and succeeds only for an administrator.
    ecr_setup = boto3.client("ecr", region_name=REGION)
    try:
        repo = ecr_setup.create_repository(repositoryName=ECR_REPO)["repository"]
        print(f"Created ECR repository: {ECR_REPO}")
    except ecr_setup.exceptions.RepositoryAlreadyExistsException:
        repo = ecr_setup.describe_repositories(repositoryNames=[ECR_REPO])["repositories"][0]
        print(f"Reusing ECR repository: {ECR_REPO}")
    ECR_URI = repo["repositoryUri"]
    print(f"ECR URI:       {ECR_URI}")

    agent_runtime = Runtime()
    agent_runtime.configure(
        entrypoint="booking_agent.py",
        execution_role=ROLE_ARN,
        ecr_repository=ECR_URI,
        auto_create_ecr=False,
        requirements_file="agent_requirements.txt",
        region=REGION,
        agent_name=RUNTIME_NAME,
        deployment_type="container",
        non_interactive=True,
    )

    print("\nLaunching agent (3-5 minutes)...")
    result = agent_runtime.launch(
        auto_update_on_conflict=True,
        env_vars={
            # Both spellings. botocore reads only AWS_DEFAULT_REGION, and the
            # workshop documents AWS_REGION.
            "AWS_REGION": REGION,
            "AWS_DEFAULT_REGION": REGION,
            "MODEL_ID": MODEL_ID,
            **NEO4J_VALUES,
        },
    )

    RUNTIME_ARN = result.agent_arn
    if not RUNTIME_ARN:
        raise RuntimeError("launch() returned no agent ARN; read the CodeBuild logs.")

    # The trailing ARN segment is the runtime ID, and it names the CloudWatch
    # log group. Printing it saves a console hunt when a smoke test below
    # produces something worth reading the logs for.
    RUNTIME_ID = RUNTIME_ARN.split("/")[-1]
    os.chdir(DEPLOY_DIR.parent)

    print(f"\nAgent deployed: {RUNTIME_ARN}")
    print(f"Runtime ID:     {RUNTIME_ID}")
    print(f"Log group:      /aws/bedrock-agentcore/runtimes/{RUNTIME_ID}-DEFAULT")

## 5. Tag what the toolkit created

The toolkit creates the ECR repository, the CodeBuild project and the Runtime,
and forwards no tags to any of them. Cleanup deletes only tagged resources, so
skipping this step leaves billable infrastructure that teardown will refuse to
touch.

Every target below is addressed by exact name or by ARN. Nothing is enumerated
and nothing is prefix-matched.

In [ ]:
if not DEPLOY_READY:
    print("Skipping tagging: nothing was deployed.")
else:
    ecr_client = boto3.client("ecr", region_name=REGION)
    codebuild_client = boto3.client("codebuild", region_name=REGION)
    agentcore = boto3.client("bedrock-agentcore-control", region_name=REGION)

    try:
        repo = ecr_client.describe_repositories(repositoryNames=[ECR_REPO])["repositories"][0]
        ecr_client.tag_resource(resourceArn=repo["repositoryArn"], tags=WORKSHOP_TAGS_KV)
        print(f"Tagged ECR repository:  {ECR_REPO}")
    except ecr_client.exceptions.RepositoryNotFoundException:
        print(f"ECR repository not found (nothing to tag): {ECR_REPO}")

    # update_project replaces the whole tag set, so merge rather than clobber
    # whatever the toolkit put there.
    projects = codebuild_client.batch_get_projects(names=[CB_PROJECT])["projects"]
    if projects:
        merged = [t for t in projects[0].get("tags", []) if t.get("key") != WORKSHOP_TAG_KEY]
        codebuild_client.update_project(name=CB_PROJECT, tags=merged + WORKSHOP_TAGS_KV_LOWER)
        print(f"Tagged CodeBuild project: {CB_PROJECT}")
    else:
        print(f"CodeBuild project not found (nothing to tag): {CB_PROJECT}")

    if not RUNTIME_ARN:
        raise RuntimeError("RUNTIME_ARN is not set. Re-run the launch cell before tagging.")
    agentcore.tag_resource(resourceArn=RUNTIME_ARN, tags=WORKSHOP_TAGS_MAP)

    # Read the tags back rather than trusting the call. A tag that did not stick
    # is a resource teardown will silently leave running.
    runtime_tags = agentcore.list_tags_for_resource(resourceArn=RUNTIME_ARN).get("tags", {})
    if runtime_tags.get(WORKSHOP_TAG_KEY) != WORKSHOP_TAG_VALUE:
        raise RuntimeError(f"Runtime tag did not stick. Read back: {runtime_tags}")
    print(f"Tagged AgentCore Runtime: {RUNTIME_ARN}")
    print("\nAll toolkit-created resources tagged and verified.")

## 6. Five smoke tests

The deployed agent is now reachable by `InvokeAgentRuntime`. Each invocation
carries its own session ID and is isolated from every other.

`grounding_result` and `command_result` come back as the tools' own structured
verdicts rather than as the model's prose. That distinction is what makes the
assertions below meaningful: a test that reads the response text passes when the
model sounds right, and a test that reads the verdict passes only when the graph
agreed.

In [ ]:
from neo4j import GraphDatabase

from workshop.contracts import MAX_GUESTS, OVER_LIMIT_GUESTS
from workshop.fixtures import (
    HERO_ADDRESS,
    HERO_NAME,
    HERO_RATING,
    HERO_SOURCE,
    load_manifest,
)
from workshop.hybrid_retrieval import Neo4jConfig

# Asks for the address explicitly, because the assertion below compares the
# exact recorded string. A question that never asks for the address cannot
# check that the answer carries it.
HERO_QUESTION = (
    f"What is the full street address and guest rating of {HERO_NAME}? "
    "Quote the address exactly as it is recorded."
)
AVAILABILITY_QUESTION = f"Does {HERO_NAME} guarantee room availability next weekend?"

# A hotel that is deliberately not in the graph. Paired with the hero hotel it
# separates "the agent refused" from "retrieval returned nothing", which look
# identical from outside if only one of the two is ever tested.
ABSENT_HOTEL_ID = "00000000-0000-4000-8000-000000000000"

# One caller-created UUID, reused for every delivery of the same reservation
# request. It is the idempotency key and the correlation identifier across the
# Runtime, the command, and the CloudWatch log lines for both.
REQUEST_ID = str(uuid.uuid4())

# Relative to today, never a hardcoded date. A fixed future date rots into the
# past and silently flips a passing check-in into a failing one.
CHECK_IN = (date.today() + timedelta(days=30)).isoformat()
CHECK_OUT = (date.today() + timedelta(days=32)).isoformat()

RESERVATION_QUERY = (
    "MATCH (r:ReservationRequest {request_id: $rid})-[:FOR_HOTEL]->(h:Hotel) "
    "RETURN r.status AS status, r.guests AS guests, h.hotel_id AS hotel_id, "
    "h.name AS hotel_name, toString(r.created_at) AS created_at"
)


def ask(prompt, runtime_arn, request_id=None, session_id=None):
    """Invoke the deployed Runtime once and print what came back.

    `runtime_arn` is a parameter rather than a closure over the module-level
    RUNTIME_ARN, so every call site shows which Runtime is being invoked. A
    helper that silently picks up whatever ARN happens to be bound is the kind
    of thing that keeps working against a Runtime you thought you tore down.
    """
    payload = {"prompt": prompt}
    if request_id is not None:
        payload["request_id"] = request_id

    client = boto3.client("bedrock-agentcore", region_name=REGION)
    response = client.invoke_agent_runtime(
        agentRuntimeArn=runtime_arn,
        runtimeSessionId=session_id or str(uuid.uuid4()),
        payload=json.dumps(payload).encode("utf-8"),
        qualifier="DEFAULT",
    )
    result = json.loads(response["response"].read())

    print(f"Q: {prompt}\n")
    print(f"A: {result.get('response')}\n")
    print(f"tools used:       {result.get('tools_used') or 'none'}")
    print(f"grounding result: {result.get('grounding_result') or 'none'}")
    print(f"command result:   {result.get('command_result') or 'none'}")
    return result


def reservation_rows(request_id):
    """Read back what the deployed agent actually wrote to the graph.

    The response text is the model's account of what happened. This is the
    graph's, and only the second one is evidence.
    """
    config = Neo4jConfig.from_environment()
    driver = GraphDatabase.driver(config.uri, auth=(config.username, config.password))
    try:
        with driver.session(database=config.database) as session:
            return [record.data() for record in session.run(RESERVATION_QUERY, rid=request_id)]
    finally:
        driver.close()


if DEPLOY_READY:
    HERO_ID = load_manifest().hotels[HERO_SOURCE]
    print(f"Hero hotel_id:  {HERO_ID}")
    print(f"Request ID:     {REQUEST_ID}")
    print(f"Stay:           {CHECK_IN} to {CHECK_OUT}")

### Test 1: the positive control

Every other test below is a refusal or a rejection, and a refusal is
indistinguishable from a broken retriever: both produce no answer. This test is
what separates them. It asserts that the deployed Runtime returns the hero
hotel's exact recorded address, so it cannot pass against an empty graph, a
dropped index, or credentials pointing somewhere else.

The compared strings come from `workshop.fixtures`, so the assertion follows the
fixture if the hero hotel ever changes.

In [ ]:
if not DEPLOY_READY:
    print("Skipping: nothing was deployed.")
else:
    hero_result = ask(HERO_QUESTION, RUNTIME_ARN)
    grounding = hero_result.get("grounding_result") or {}
    top = grounding.get("top_evidence") or {}

    assert grounding.get("answerable") is True, grounding
    assert HERO_ID in (grounding.get("evidence_ids") or []), grounding

    # Exact values, never a substring and never a non-empty check. "Cairo"
    # appears in several documents and `is not None` passes on anything, so
    # either one would go green against a graph that had lost the hero hotel.
    assert top.get("hotel_id") == HERO_ID, top
    assert top.get("hotel_name") == HERO_NAME, top
    assert top.get("address") == HERO_ADDRESS, top
    assert top.get("guest_rating") == HERO_RATING, top

    # The tool returned the right facts. This asserts the model also answered
    # with them, which covers the whole path rather than just the retriever.
    assert HERO_ADDRESS in (hero_result.get("response") or ""), hero_result.get("response")
    print(f"\nPASS: returned the recorded address {HERO_ADDRESS} and rating {HERO_RATING}.")

### Test 2: a hotel that does not exist

The negative half of the pair. A hotel ID that is not in the graph never
reaches the reservation command, because the agent may only pass an ID that
grounded search returned. Nothing is written.

In [ ]:
if not DEPLOY_READY:
    print("Skipping: nothing was deployed.")
else:
    absent_request_id = str(uuid.uuid4())
    absent = ask(
        f"Create a reservation request at hotel {ABSENT_HOTEL_ID} from {CHECK_IN} "
        f"to {CHECK_OUT} for 2 guests.",
        RUNTIME_ARN,
        request_id=absent_request_id,
    )
    absent_command = absent.get("command_result") or {}

    assert absent_command.get("status") != "accepted", absent_command
    assert reservation_rows(absent_request_id) == [], "wrote a request for a hotel that does not exist"
    assert ABSENT_HOTEL_ID not in ((absent.get("grounding_result") or {}).get("evidence_ids") or [])
    print("\nPASS: refused an ungrounded hotel_id, nothing written.")

### Test 3: a question the graph cannot answer

The graph holds hotel knowledge, not live inventory, so the tool decides this
one rather than the model. The abstention is paired with its own positive
control: the same verdict still carries the hero hotel's exact address, which
proves retrieval was working at the moment the agent declined to answer.

In [ ]:
if not DEPLOY_READY:
    print("Skipping: nothing was deployed.")
else:
    availability_result = ask(AVAILABILITY_QUESTION, RUNTIME_ARN)
    grounding = availability_result.get("grounding_result") or {}

    assert grounding.get("answerable") is False, grounding
    assert grounding.get("missing_fact") == "live_room_availability", grounding

    # The control that makes the abstention mean something. Without it this
    # cell passes just as happily against an index that returns nothing.
    top = grounding.get("top_evidence") or {}
    assert top.get("address") == HERO_ADDRESS, top
    print("\nPASS: abstained on availability while retrieval was demonstrably live.")

### Test 4: an over-limit request is rejected with no write

The maximum-guests rule lives in Neo4j and is enforced inside the same
transaction as the write. The request names the hotel rather than its ID, so the
agent has to ground the `hotel_id` through search first, and the assertion below
checks it grounded the right one.

In [ ]:
if not DEPLOY_READY:
    print("Skipping: nothing was deployed.")
else:
    rejected = ask(
        f"Create a reservation request at the hotel named {HERO_NAME} from "
        f"{CHECK_IN} to {CHECK_OUT} for {OVER_LIMIT_GUESTS} guests.",
        RUNTIME_ARN,
        request_id=REQUEST_ID,
    )
    command = rejected.get("command_result") or {}

    assert command.get("status") == "rejected", command
    assert command.get("reason_code") == "max_guests_exceeded", command
    # Grounded from search, not from the prompt, which never contained an ID.
    assert command.get("hotel_id") == HERO_ID, command
    assert reservation_rows(REQUEST_ID) == [], "an over-limit request was written"
    print(f"\nPASS: rejected with reason_code={command.get('reason_code')}, no node written.")

### Test 5: a valid request is recorded, and safe to retry

Reusing the same `request_id`, a request within the limit is accepted and one
`ReservationRequest` is linked to the hero hotel. Re-delivering the identical
request returns the existing record with `duplicate=true` and creates no second
node, so a retried invocation is safe.

In [ ]:
if not DEPLOY_READY:
    print("Skipping: nothing was deployed.")
else:
    reservation_prompt = (
        f"Create a reservation request at the hotel named {HERO_NAME} from "
        f"{CHECK_IN} to {CHECK_OUT} for {MAX_GUESTS} guests."
    )
    accepted = ask(reservation_prompt, RUNTIME_ARN, request_id=REQUEST_ID)
    command = accepted.get("command_result") or {}
    assert command.get("status") == "accepted", command
    assert command.get("hotel_id") == HERO_ID, command

    rows = reservation_rows(REQUEST_ID)
    assert len(rows) == 1, rows
    assert rows[0]["hotel_id"] == HERO_ID, rows
    assert rows[0]["guests"] == MAX_GUESTS, rows
    print(f"\nGraph says: {json.dumps(rows[0], indent=2)}")

    # A separate session ID, because a replay from a different caller is the
    # case that matters. Sharing the session would let conversation history,
    # rather than the idempotency key, explain a correct answer.
    replay = ask(reservation_prompt, RUNTIME_ARN, request_id=REQUEST_ID)
    replay_command = replay.get("command_result") or {}
    assert replay_command.get("duplicate") is True, replay_command
    assert len(reservation_rows(REQUEST_ID)) == 1, "replay created a second node"
    print("\nPASS: one node written, replay returned duplicate=true, still one node.")

## What is running now

```
InvokeAgentRuntime
        |
        v
+---------------------------+
|  AgentCore Runtime        |
|  GraphRagBookingAgent     |
|                           |
|  booking_agent.py         |
|   +- search_hotel_knowledge  --> Neo4j hybrid retrieval
|   +- create_reservation      --> Neo4j write, rule enforced in-transaction
|   +- BedrockModel            --> Claude on Amazon Bedrock
+---------------------------+
```

The agent reasons exactly as it did in Module 3.2. What moved is the boundary:
the container holds the Neo4j credentials, AgentCore holds the container, and a
caller needs `bedrock-agentcore:InvokeAgentRuntime` rather than a Python
environment.

Two properties survived the move, and they are the ones worth checking after any
deployment:

- **Retrieval held.** The hero question came back with the exact recorded
  address, so every refusal below it is a refusal and not an empty index.
- **Abstention held.** The availability question returned `answerable: false`
  from the tool, not a hedge from the model.
- **The rule held.** The 15-guest request was refused by Neo4j inside the write
  transaction, and the graph confirms no node was created.

:::alert{type="info" header="Leave it running for Module 6, or tear it down"}
Module 6 does not depend on this Runtime. The cleanup notebook removes the
Runtime, the ECR repository, the CodeBuild project and the execution role by
their `WorkshopResource` tag.
:::